In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as mticker
from scipy.ndimage import gaussian_filter1d
import tensorflow as tf

In [2]:
import simclr_models, simclr_utitlities

In [3]:
def get_sincconv_layer(model, layer_name="sincconv"):
    for layer in model.layers:
        if layer.name == layer_name:
            return layer
    # recurse into sub-models
    for layer in model.layers:
        if hasattr(layer, "layers"):
            result = get_sincconv_layer(layer, layer_name)
            if result is not None:
                return result
    return None

In [4]:
model = tf.keras.models.load_model("20260604-064309_simclr_ms_j.keras", custom_objects={'SincConv1D': simclr_models.SincConv1D})

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

sinc_layer = model.get_layer("sincconv")
num_channels = sinc_layer._num_channels
num_filters = sinc_layer.num_filters
sample_rate = sinc_layer.sample_rate
nyquist = sample_rate / 2.0

sinc_weights = sinc_layer.weights

for w in sinc_weights:
    print(f"Name: {w.name} | Shape: {w.shape}")
    print(f"Values: {w.numpy()}")

Name: f1 | Shape: (3, 4)
Values: [[ 1.3064994   0.10732682 -0.24889848  6.406944  ]
 [-0.67892134  0.07933081  0.7820558  -5.7142186 ]
 [-0.33572575  0.35452768  0.25258046 -7.3865867 ]]
Name: band | Shape: (3, 4)
Values: [[ 0.57912695  0.1453345   0.6350283   1.7736663 ]
 [ 0.47343254  0.31336707 -0.4846511   1.0707451 ]
 [ 0.26815075  0.38118425  0.38821632  1.0223516 ]]


In [ ]:
min_low_hz = 0.1 / nyquist
min_band_hz = 0.5 / nyquist

w_dict = {w.name.split(':')[0]: w.numpy() for w in sinc_weights}
f1_raw = w_dict['f1']
band_raw = w_dict['band']

channels, total_filters = f1_raw.shape

for c in range(channels):
    print(f"\nChannel {c}:")
    for f in range(total_filters):
        
        f1_normalized = min_low_hz + np.abs(f1_raw[c, f])
        f2_normalized = f1_normalized + min_band_hz + np.abs(band_raw[c, f])

        f_low_hz = f1_normalized * nyquist
        f_high_hz = f2_normalized * nyquist
        f_band_hz = f_high_hz - f_low_hz
        
        print(f"Filter {f}: Passband = [{f_low_hz:8.2f} Hz to {f_high_hz:8.2f} Hz] | Bandwidth = {f_band_hz:7.2f} Hz")


Channel 0:
Filter 0: Passband = [   32.76 Hz to    47.74 Hz] | Bandwidth =   14.98 Hz
Filter 1: Passband = [    2.78 Hz to     6.92 Hz] | Bandwidth =    4.13 Hz
Filter 2: Passband = [    6.32 Hz to    22.70 Hz] | Bandwidth =   16.38 Hz
Filter 3: Passband = [  160.27 Hz to   205.12 Hz] | Bandwidth =   44.84 Hz

Channel 1:
Filter 0: Passband = [   17.07 Hz to    29.41 Hz] | Bandwidth =   12.34 Hz
Filter 1: Passband = [    2.08 Hz to    10.42 Hz] | Bandwidth =    8.33 Hz
Filter 2: Passband = [   19.65 Hz to    32.27 Hz] | Bandwidth =   12.62 Hz
Filter 3: Passband = [  142.96 Hz to   170.22 Hz] | Bandwidth =   27.27 Hz

Channel 2:
Filter 0: Passband = [    8.49 Hz to    15.70 Hz] | Bandwidth =    7.20 Hz
Filter 1: Passband = [    8.96 Hz to    18.99 Hz] | Bandwidth =   10.03 Hz
Filter 2: Passband = [    6.41 Hz to    16.62 Hz] | Bandwidth =   10.21 Hz
Filter 3: Passband = [  184.76 Hz to   210.82 Hz] | Bandwidth =   26.06 Hz
